# 22g — Monte-Carlo at the TRUE experiment values: all three estimators vs the real data

**Question (Anuar).** If we generate simulated data at an experiment's **true** parameters and analyse it
with **the same estimator Gregor used on the real data** (binned Voigt least-squares), the simulated cloud
should look like the real cloud. In 22e/22f it did not — but 22e **pinned** the count-noise condition
(sigma_prop=9.851, lambda=2.593 = 1nW T60) for the whole grid, so 13 of 14 experiments were compared against
the wrong noise. This notebook removes that confound.

**What it does.** For each of the 14 experiments, generate **N_MC = 1000 runs at that experiment's own true
values** (`mu_true, gamma_true, sigma_prop, lam`), and fit **every run with all three estimators on the same
photons**:

1. `ours: Lorentzian MLE` — unbinned MLE (the current pipeline),
2. `ours: pseudo-Voigt` — unbinned, Lorentzian + **fixed eta = 0.2** Gaussian admixture,
3. `Gregor: binned Voigt LSQ` — bin at 2.2 MHz, `lmfit` Voigt + constant least-squares (verbatim).

Then overlay the **real** `(FWHM, sigma_fit)` points of that experiment.

**Companion run data.** The raw per-run clouds are saved to `data/processed/22g_clouds.npz` (0.8 MB) — the
data 22e did not keep. The notebook **loads that npz by default** and only regenerates the Monte-Carlo if it
is missing (or with env `NB_RECOMPUTE=1`), so figures/analysis can be iterated without the ~21 min run.

In [1]:
# ============================================================
# 22g — imports
# ============================================================
import math, time, os, sys
import numpy as np
import torch
import multiprocessing as _mp
from concurrent.futures import ProcessPoolExecutor as _PPE
from scipy.ndimage import gaussian_filter

torch.set_default_dtype(torch.float32)
for _p in [os.getcwd(), os.path.join(os.getcwd(), '..'), os.path.join(os.getcwd(), '..', '..')]:
    if os.path.isdir(os.path.join(_p, 'src')):
        sys.path.insert(0, _p); REPO_ROOT = _p; break
os.chdir(REPO_ROOT)

from src.fitting import fit_profile, fwhm_from_theta, nll
from src.samplers import draw_fixed_noise, build_photons
from src.implicit import compute_fwhm_and_dgamma
from src import fitting_lmfit as FL

import plotly
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
pio.templates.default = 'plotly_white'
pio.renderers.default = os.environ.get('PLOTLY_RENDERER', 'vscode')
print('Imports OK | plotly', plotly.__version__, '| lmfit', FL.lmfit.__version__)


Imports OK | plotly 7.1.0 | lmfit 1.3.4


In [2]:
# ============================================================
# 22g — EXPERIMENTS (true values from Gregor's fits; identical to 22a/22b/22c)
# ============================================================
EXPERIMENTS = [
    dict(name='1nW Trans05',  power='1nW', mu_true=9.393, sigma_prop=2.576, lam=2.232, gamma_true=8.5, n_target=61, data_file='data/raw_data/fwhm_1nW_240221/fwhm_1nW_240221SIL_Puppy_hindleg_red1nW_Top20nW_Trans05.txt'),
    dict(name='1nW Trans10',  power='1nW', mu_true=12.372, sigma_prop=3.445, lam=2.122, gamma_true=8.5, n_target=358, data_file='data/raw_data/fwhm_1nW_240221/fwhm_1nW_240221SIL_Puppy_hindleg_red1nW_Top20nW_Trans10.txt'),
    dict(name='1nW Trans20',  power='1nW', mu_true=17.316, sigma_prop=4.141, lam=2.286, gamma_true=8.5, n_target=1138, data_file='data/raw_data/fwhm_1nW_240221/fwhm_1nW_240221SIL_Puppy_hindleg_red1nW_Top20nW_Trans20.txt'),
    dict(name='1nW Trans40',  power='1nW', mu_true=38.405, sigma_prop=7.198, lam=2.351, gamma_true=8.5, n_target=2428, data_file='data/raw_data/fwhm_1nW_240221/fwhm_1nW_240221SIL_Puppy_hindleg_red1nW_Top20nW_Trans40.txt'),
    dict(name='1nW Trans60',  power='1nW', mu_true=61.374, sigma_prop=9.851, lam=2.593, gamma_true=8.5, n_target=2424, data_file='data/raw_data/fwhm_1nW_240221/fwhm_1nW_240221SIL_Puppy_hindleg_red1nW_Top20nW_Trans60.txt'),
    dict(name='1nW Trans80',  power='1nW', mu_true=79.365, sigma_prop=12.627, lam=2.758, gamma_true=8.5, n_target=2487, data_file='data/raw_data/fwhm_1nW_240221/fwhm_1nW_240221SIL_Puppy_hindleg_red1nW_Top20nW_Trans80.txt'),
    dict(name='1nW Trans100', power='1nW', mu_true=70.817, sigma_prop=17.221, lam=2.636, gamma_true=8.5, n_target=2455, data_file='data/raw_data/fwhm_1nW_240221/fwhm_1nW_240221SIL_Puppy_hindleg_red1nW_Top20nW_Trans100.txt'),
    dict(name='3nW Trans05',  power='3nW', mu_true=13.204, sigma_prop=3.724, lam=2.186, gamma_true=14.1, n_target=252, data_file='data/raw_data/fwhm_3nW_210221/fwhm_3nW_210221SIL_Puppy_hindleg_red3nW_Top20nW_Trans05.txt'),
    dict(name='3nW Trans10',  power='3nW', mu_true=24.476, sigma_prop=5.639, lam=2.158, gamma_true=14.1, n_target=1572, data_file='data/raw_data/fwhm_3nW_210221/fwhm_3nW_210221SIL_Puppy_hindleg_red3nW_Top20nW_Trans10.txt'),
    dict(name='3nW Trans20',  power='3nW', mu_true=34.279, sigma_prop=8.319, lam=2.264, gamma_true=14.1, n_target=2171, data_file='data/raw_data/fwhm_3nW_210221/fwhm_3nW_210221SIL_Puppy_hindleg_red3nW_Top20nW_Trans20.txt'),
    dict(name='3nW Trans40',  power='3nW', mu_true=84.892, sigma_prop=24.013, lam=2.475, gamma_true=14.1, n_target=3742, data_file='data/raw_data/fwhm_3nW_210221/fwhm_3nW_210221SIL_Puppy_hindleg_red3nW_Top20nW_Trans40.txt'),
    dict(name='3nW Trans60',  power='3nW', mu_true=103.203, sigma_prop=23.95, lam=2.741, gamma_true=14.1, n_target=2541, data_file='data/raw_data/fwhm_3nW_210221/fwhm_3nW_210221SIL_Puppy_hindleg_red3nW_Top20nW_Trans60.txt'),
    dict(name='3nW Trans80',  power='3nW', mu_true=137.537, sigma_prop=32.107, lam=2.911, gamma_true=14.1, n_target=2508, data_file='data/raw_data/fwhm_3nW_210221/fwhm_3nW_210221SIL_Puppy_hindleg_red3nW_Top20nW_Trans80.txt'),
    dict(name='3nW Trans100', power='3nW', mu_true=175.707, sigma_prop=40.975, lam=3.087, gamma_true=14.1, n_target=2516, data_file='data/raw_data/fwhm_3nW_210221/fwhm_3nW_210221SIL_Puppy_hindleg_red3nW_Top20nW_Trans100.txt'),
]
print(len(EXPERIMENTS), 'experiments')


14 experiments


In [3]:
# ============================================================
# 22g CONFIG — every experiment uses ITS OWN true (mu, gamma, sigma_prop, lam)
# ============================================================
N_MC    = 1000                # runs per experiment (same as 22b/22c)
SEED    = 42
N_WORKERS = 4
BIN_WIDTH, WINDOW, MIN_COUNTS = 2.2, 75.0, 3     # Gregor's estimator settings

X_RANGE = (0.0, 70.0); Y_RANGE = (0.0, 40.0)     # FWHM, sigma_fit (MHz)
NPZ_OUT  = 'data/processed/22g_clouds.npz'
HTML_OUT = 'notebooks/22_distribution_dynamics/22g-mc-at-truth-3-estimators.html'

POWERS   = ['1nW', '3nW']
TRANS    = ['Trans05', 'Trans10', 'Trans20', 'Trans40', 'Trans60', 'Trans80', 'Trans100']
CLEAN_T  = ['Trans40', 'Trans60', 'Trans80', 'Trans100']      # high-T: real cloud is clean
LOW_T    = ['Trans05', 'Trans10', 'Trans20']                  # low-T: real cloud is degenerate (see verdict)

EST_NAMES  = {'lorentzian': 'ours: Lorentzian MLE', 'pseudo_voigt': 'ours: pseudo-Voigt fit',
              'gregor': 'Gregor: binned Voigt LSQ'}
EST_COLORS = {'lorentzian': '#1d3557', 'pseudo_voigt': '#2a9d8f', 'gregor': '#e76f51'}
EXP_COLORS = ['#e6194B', '#4363d8', '#3cb44b', '#f58231', '#911eb4', '#f032e6', '#469990',
              '#9A6324', '#800000', '#808000', '#000075', '#008080', '#8B4513', '#c71585']

SMOKE = os.environ.get('NB_SMOKE') == '1'
if SMOKE:
    N_MC = 60; EXPERIMENTS = EXPERIMENTS[:2]; TRANS = TRANS[:2]
print('config: N_MC=%d, workers=%d, exps=%d, SMOKE=%s' % (N_MC, N_WORKERS, len(EXPERIMENTS), SMOKE))


config: N_MC=1000, workers=4, exps=14, SMOKE=False


In [4]:
# ============================================================
# 22g — estimators + one-experiment Monte-Carlo (all three on the SAME photons)
# ============================================================
def _fit_lor(ph):  return fit_profile(ph, n_iters=80, model='lorentzian',   uniform_bg=False)
def _fwhm_lor(th): return fwhm_from_theta(th, model='lorentzian')
def _nll_lor(th, ph): return nll(th, ph, model='lorentzian', uniform_bg=False)

def _fit_pv(ph):   return fit_profile(ph, n_iters=80, model='pseudo-voigt', uniform_bg=False)
def _fwhm_pv(th):  return fwhm_from_theta(th, model='pseudo-voigt')
def _nll_pv(th, ph): return nll(th, ph, model='pseudo-voigt', uniform_bg=False)

def lmfit_extract(photons_np, bin_width=BIN_WIDTH, window=WINDOW):
    '''Gregor's estimator (verbatim from 21d/22c): bin, LSQ Voigt+constant, return (FWHM, stderr).'''
    lo, hi = -window / 2.0, window / 2.0
    edges = np.arange(lo, hi + bin_width, bin_width)
    counts, edges = np.histogram(photons_np, bins=edges)
    if counts.size == 0 or counts.max() < MIN_COUNTS:
        return None, None
    x = 0.5 * (edges[:-1] + edges[1:])
    m = FL.voigt
    p = m.make_params()
    p['amplitude'].set(value=max(float(counts.sum()), 1.0), min=0.0)
    p['center'].set(value=float(x[int(np.argmax(counts))]), min=lo, max=hi)
    p['sigma'].set(value=3.0, min=0.05, max=50.0)
    p['gamma'].set(value=3.0, min=0.05, max=100.0)
    p['c'].set(value=float(np.median(counts)), min=0.0)
    try:
        out = m.fit(counts, p, x=x)
    except Exception:
        return None, None
    fw = out.params['fwhm'].value; se = out.params['fwhm'].stderr
    fw = float(fw) if fw is not None and np.isfinite(fw) else None
    se = float(se) if se is not None and np.isfinite(se) else None
    return fw, se

def _mc_lorentzian(args):
    gamma, u, b = args
    return compute_fwhm_and_dgamma(gamma, u, b, _fit_lor, _fwhm_lor, _nll_lor, n_params=2)

def _mc_pseudo_voigt(args):
    gamma, u, b = args
    return compute_fwhm_and_dgamma(gamma, u, b, _fit_pv, _fwhm_pv, _nll_pv, n_params=3)

def _mc_gregor(args):
    gamma, u, b = args
    ph = build_photons(torch.tensor(float(gamma)), torch.as_tensor(u, dtype=torch.float32),
                       torch.as_tensor(b, dtype=torch.float32)).numpy()
    fw, se = lmfit_extract(ph)
    return (np.nan if fw is None else fw, np.nan if se is None else se)

_WORKER = {'lorentzian': _mc_lorentzian, 'pseudo_voigt': _mc_pseudo_voigt, 'gregor': _mc_gregor}
ESTS3 = ('lorentzian', 'pseudo_voigt', 'gregor')
def _init_worker(): torch.set_num_threads(1)

def cloud_at_truth(exp, pool):
    '''N_MC runs at the experiment's OWN (mu_true, gamma_true, sigma_prop, lam); all three
    estimators see the SAME frozen photons (paired comparison).'''
    mu, sp, lam, ga = exp['mu_true'], exp['sigma_prop'], exp['lam'], exp['gamma_true']
    rng = np.random.default_rng(SEED)
    tasks = []
    for _ in range(N_MC):
        u, b, n = draw_fixed_noise(mu, sp, lam, rng)
        tasks.append((ga, u.numpy(), b.numpy()))
    out = {}
    for ek in ESTS3:
        res = list(pool.map(_WORKER[ek], tasks, chunksize=8))
        out[ek] = (np.array([r[0] for r in res], float), np.array([r[1] for r in res], float))
    return out

def load_real(exp):
    d = np.genfromtxt(exp['data_file'])
    f = d[:, 0] * 1000.0; e = d[:, 1] * 1000.0
    ok = ~np.isnan(f) & ~np.isnan(e) & (f > 0)
    flt = ok & (e / f < 10.0)
    return f[flt], e[flt]

print('estimators + worker ready')


estimators + worker ready


In [5]:
# ============================================================
# 22g — RUN (or LOAD the companion data): clouds at truth for all 14 experiments
#   cached: regenerates only if data/processed/22g_clouds.npz is absent, or with NB_RECOMPUTE=1
# ============================================================
CLOUDS = {}     # name -> {est: (f, s)}
REAL   = {}     # name -> (f, s)

def _payload():
    pay = {}
    for name, cd in CLOUDS.items():
        for ek, (f, s) in cd.items():
            pay[f'{name}|{ek}|f'] = f; pay[f'{name}|{ek}|s'] = s
        rf, rs = REAL[name]; pay[f'{name}|real|f'] = rf; pay[f'{name}|real|s'] = rs
    return pay

if (not SMOKE) and os.path.exists(NPZ_OUT) and os.environ.get('NB_RECOMPUTE') != '1':
    _d = np.load(NPZ_OUT, allow_pickle=True)
    for exp in EXPERIMENTS:
        n = exp['name']
        CLOUDS[n] = {ek: (_d[f'{n}|{ek}|f'].astype(float), _d[f'{n}|{ek}|s'].astype(float)) for ek in ESTS3}
        REAL[n]   = (_d[f'{n}|real|f'].astype(float), _d[f'{n}|real|s'].astype(float))
    print(f'loaded companion run data {NPZ_OUT}  (N_MC={N_MC}; set NB_RECOMPUTE=1 to regenerate)')
else:
    t0 = time.time()
    with _PPE(max_workers=N_WORKERS, mp_context=_mp.get_context('fork'), initializer=_init_worker) as pool:
        for exp in EXPERIMENTS:
            te = time.time()
            CLOUDS[exp['name']] = cloud_at_truth(exp, pool)
            REAL[exp['name']] = load_real(exp)
            n_ok = {ek: int(np.isfinite(v[0]).sum()) for ek, v in CLOUDS[exp['name']].items()}
            print(f"  {exp['name']:13s} done in {(time.time()-te)/60:5.1f} min | n_ok {n_ok}", flush=True)
    print(f'\nprecompute total: {(time.time()-t0)/60:.1f} min  |  runs/exp: {N_MC}')
    if not SMOKE:
        np.savez_compressed(NPZ_OUT, **_payload())
        print('saved', NPZ_OUT, f'({os.path.getsize(NPZ_OUT)/1e6:.2f} MB)')


loaded companion run data data/processed/22g_clouds.npz  (N_MC=1000; set NB_RECOMPUTE=1 to regenerate)


In [6]:
# ============================================================
# 22g — FIG 1: per-experiment cloud at truth (3 estimators) + real data
#   2 columns (1nW / 3nW) x one row per transparency
# ============================================================
def _dens_xy(f, s, xr=X_RANGE, yr=Y_RANGE, nx=40, ny=40):
    ok = np.isfinite(f) & np.isfinite(s) & (f >= xr[0]) & (f <= xr[1]) & (s >= yr[0]) & (s <= yr[1])
    H, xe, ye = np.histogram2d(f[ok], s[ok], bins=[nx, ny], range=[xr, yr])
    Z = gaussian_filter(H.T, 1.1); Z = Z / max(Z.max(), 1e-12)
    return 0.5 * (xe[:-1] + xe[1:]), 0.5 * (ye[:-1] + ye[1:]), Z.astype(float)   # Z in [0,1], shape (ny,nx)

# --- iso-contour extraction (marching squares via matplotlib; computation only, no figure shown/saved) ---
from matplotlib.figure import Figure
from matplotlib.backends.backend_agg import FigureCanvasAgg

BAND_LEVELS = (0.15, 0.35, 0.55, 0.75)
BAND_ALPHAS = (0.08, 0.11, 0.14, 0.17)

def cloud_contours(xc, yc, Z01, levels=BAND_LEVELS):
    '''Return [(band_index, level, [closed_loop_xy, ...]), ...] for the normalized density Z01 (ny, nx).'''
    if Z01.max() <= 0:
        return []
    _f = Figure(); FigureCanvasAgg(_f)
    _ax = _f.add_subplot(111)
    cs = _ax.contour(xc, yc, Z01, levels=list(levels))
    return [(i, float(lev), [seg for seg in segs if seg.shape[0] >= 3])
            for i, (lev, segs) in enumerate(zip(cs.levels, cs.allsegs))]

def _rgba(hexcol, a):
    h = hexcol.lstrip('#'); return f'rgba({int(h[0:2],16)},{int(h[2:4],16)},{int(h[4:6],16)},{a})'

EXP_IDX = {e['name']: k for k, e in enumerate(EXPERIMENTS)}
EXP_BY  = {e['name']: e for e in EXPERIMENTS}
fig = make_subplots(rows=len(TRANS), cols=len(POWERS), horizontal_spacing=0.07, vertical_spacing=0.045,
                    subplot_titles=[(f'{p} {t}   (mu={EXP_BY[p + " " + t]["mu_true"]:g}, gamma={EXP_BY[p + " " + t]["gamma_true"]:g})'
                                     if (p + " " + t) in EXP_BY else f'{p} {t}')
                                    for t in TRANS for p in POWERS])
seen = set()
def _first(grp):
    s = grp not in seen; seen.add(grp); return s

# ---- pass 1: ALL real-data points (bottom layer — the contours then draw OVER them) ----
for r, t in enumerate(TRANS, start=1):
    for c, p in enumerate(POWERS, start=1):
        name = f'{p} {t}'
        if name not in CLOUDS: continue
        rf, rs = REAL[name]
        fig.add_trace(go.Scatter(x=rf, y=rs, mode='markers',
            marker=dict(size=4, color=EXP_COLORS[EXP_IDX[name]], opacity=0.9,
                        line=dict(width=0.4, color='rgba(20,20,20,0.55)')), hoverinfo='skip',
            legendgroup='real_' + name, name='real ' + name, showlegend=_first('real_' + name)), r, c)

# ---- pass 2: ALL contour bands, as SCATTER polygons so they sit OVER the points ----
#   (go.Contour lives in a fixed plotly layer BELOW scatterlayer, so trace order cannot lift it)
for r, t in enumerate(TRANS, start=1):
    for c, p in enumerate(POWERS, start=1):
        name = f'{p} {t}'
        if name not in CLOUDS: continue
        for ek in ESTS3:
            f, s = CLOUDS[name][ek]
            xc, yc, Z = _dens_xy(f, s)
            for bi, lev, loops in cloud_contours(xc, yc, Z):
                for lp in loops:
                    fig.add_trace(go.Scatter(
                        x=lp[:, 0], y=lp[:, 1], mode='lines', fill='toself',
                        fillcolor=_rgba(EST_COLORS[ek], BAND_ALPHAS[bi]),
                        line=dict(color=EST_COLORS[ek], width=1.0),
                        hoverinfo='skip', legendgroup='est_' + ek, name=EST_NAMES[ek],
                        showlegend=_first('est_' + ek)), r, c)
        fig.update_xaxes(range=X_RANGE, showticklabels=(r == len(TRANS)), row=r, col=c)
        fig.update_yaxes(range=Y_RANGE, showticklabels=(c == 1), row=r, col=c)

# ---- pass 3: the REAL experiment's own density contours (band-vs-band, fair comparison) ----
for r, t in enumerate(TRANS, start=1):
    for c, p in enumerate(POWERS, start=1):
        name = f'{p} {t}'
        if name not in CLOUDS: continue
        rf, rs = REAL[name]
        xc, yc, Z = _dens_xy(rf, rs)
        for bi, lev, loops in cloud_contours(xc, yc, Z):
            for lp in loops:
                fig.add_trace(go.Scatter(
                    x=lp[:, 0], y=lp[:, 1], mode='lines', fill='toself',
                    fillcolor='rgba(0,0,0,0.05)',
                    line=dict(color='#111111', width=1.4, dash='dash'),
                    hoverinfo='skip', legendgroup='real_contour',
                    name='REAL data (density contour)', showlegend=_first('real_contour')), r, c)
fig.update_layout(height=340 * len(TRANS) + 120, width=640 * len(POWERS) + 160,
    legend=dict(groupclick='togglegroup', font=dict(size=11), itemsizing='constant'),
    margin=dict(l=80, r=20, t=80, b=45),
    title_text='22g — Monte-Carlo at the TRUE experiment values  |  contours = 3 estimators, dots = real data  |  '
               'columns = power, rows = transparency')
fig.update_annotations(font=dict(size=11))
fig.add_annotation(text='<b>1nW</b>', xref='paper', yref='paper', x=0.25, y=1.022, showarrow=False, font=dict(size=15))
fig.add_annotation(text='<b>3nW</b>', xref='paper', yref='paper', x=0.75, y=1.022, showarrow=False, font=dict(size=15))
CONFIG = {'scrollZoom': True, 'displaylogo': False, 'responsive': True}
print('FIG1 traces:', len(fig.data))
fig.show(config=CONFIG)


FIG1 traces: 254


In [7]:
# ============================================================
# 22g — quantitative comparison, channel by channel (robust medians, high-T clean subset)
# ============================================================
def _med(v):
    v = np.asarray(v, float); v = v[np.isfinite(v)]
    return float(np.median(v)) if v.size else np.nan
def _iqr(v):
    v = np.asarray(v, float); v = v[np.isfinite(v)]
    return float(np.percentile(v, 75) - np.percentile(v, 25)) if v.size else np.nan

CLEAN = [f'{p} {t}' for t in CLEAN_T for p in POWERS]      # 8 high-T experiments
LOWT  = [f'{p} {t}' for t in LOW_T  for p in POWERS]

print('Per-experiment   [FWHM med ± IQR   /   sigma_fit med ± IQR]   (MHz)')
print(f'{"experiment":13s}| {"REAL":>21s} |' + ''.join(f' {EST_NAMES[e][:20]:>21s} |' for e in ESTS3))
for t in TRANS:
    for p in POWERS:
        name = f'{p} {t}'
        if name not in CLOUDS: continue
        rf, rs = REAL[name]
        line = f'{name:13s}| {_med(rf):7.2f}±{_iqr(rf):5.2f} / {_med(rs):5.2f}±{_iqr(rs):5.2f} |'
        for ek in ESTS3:
            f, s = CLOUDS[name][ek]
            line += f' {_med(f):7.2f}±{_iqr(f):5.2f} / {_med(s):5.2f}±{_iqr(s):5.2f} |'
        print(line)

def _rel(ek, chan, names):
    errs = []
    for n in names:
        if n not in CLOUDS: continue
        if chan == 'f_med': a, b = _med(CLOUDS[n][ek][0]), _med(REAL[n][0])
        elif chan == 'f_iqr': a, b = _iqr(CLOUDS[n][ek][0]), _iqr(REAL[n][0])
        else:                 a, b = _med(CLOUDS[n][ek][1]), _med(REAL[n][1])
        if np.isfinite(a) and np.isfinite(b) and b > 0: errs.append(abs(a - b) / b)
    return float(np.mean(errs)) if errs else np.nan

print(f'\nMean |sim - real| / real   on the CLEAN high-T subset ({len(CLEAN)} exps, {', '.join(CLEAN_T)}):')
print(f'{"estimator":26s}  {"FWHM median":>12s}  {"FWHM IQR":>10s}  {"sigma_fit median":>16s}')
for ek in ESTS3:
    print(f'  {EST_NAMES[ek]:24s}  {_rel(ek,"f_med",CLEAN):11.1%}  {_rel(ek,"f_iqr",CLEAN):9.1%}  '
          f'{_rel(ek,"s_med",CLEAN):15.1%}')
print('\n(For reference, same on the LOW-T degenerate subset — the real cloud there is dominated by')
print(' failed/ill-conditioned fits, so no simulator can match it:)')
for ek in ESTS3:
    print(f'  {EST_NAMES[ek]:24s}  {_rel(ek,"f_med",LOWT):11.1%}  {_rel(ek,"f_iqr",LOWT):9.1%}  '
          f'{_rel(ek,"s_med",LOWT):15.1%}')


Per-experiment   [FWHM med ± IQR   /   sigma_fit med ± IQR]   (MHz)
experiment   |                  REAL |  ours: Lorentzian MLE |  ours: pseudo-Voigt f |  Gregor: binned Voigt |
1nW Trans05  |    3.74±22.11 /  4.63± 8.27 |   22.30±17.02 /  3.12± 2.82 |   42.77±37.40 /  6.09± 7.69 |    6.00± 0.00 /  3.09± 3.13 |
3nW Trans05  |   16.74±28.09 /  4.24± 5.87 |   32.36±20.98 /  3.51± 3.00 |   58.80±38.88 /  6.71± 6.65 |    6.00± 0.00 /  3.89± 3.17 |
1nW Trans10  |    6.38±14.70 /  2.30± 4.78 |   20.86±13.95 /  2.28± 1.97 |   38.63±27.23 /  4.51± 4.89 |    6.00± 1.15 /  3.05± 2.95 |
3nW Trans10  |   19.41±19.29 /  2.52± 3.22 |   30.19±12.94 /  1.77± 1.10 |   53.11±24.76 /  3.42± 2.56 |    6.00± 8.43 /  4.35± 4.61 |
1nW Trans20  |   10.78±14.02 /  1.65± 2.22 |   19.60±10.63 /  1.58± 1.08 |   35.77±20.43 /  3.15± 2.55 |    6.00± 6.82 /  3.20± 3.47 |
3nW Trans20  |   26.04±17.96 /  2.73± 3.05 |   30.09±10.86 /  1.30± 0.71 |   52.18±21.05 /  2.58± 1.67 |    7.86±14.90 /  4.96± 4.99 |
1nW Trans40

In [8]:
# ============================================================
# 22g — write the standalone HTML
# ============================================================
os.makedirs(os.path.dirname(HTML_OUT), exist_ok=True)
fig.write_html(HTML_OUT, include_plotlyjs='cdn', config=CONFIG)
print('wrote', HTML_OUT, f'({os.path.getsize(HTML_OUT)/1e6:.1f} MB)')


wrote notebooks/22_distribution_dynamics/22g-mc-at-truth-3-estimators.html (0.9 MB)


## Verdict — the matched estimator does NOT reproduce the real cloud (and is worst on sigma_fit)

Setup: **N_MC = 1000** runs per experiment, all at that experiment's **own** true
`(mu, gamma, sigma_prop, lam)`, with all three estimators applied to the **same** photons. This is the clean
version of "same estimator as the real data was produced with".

**Clean high-T subset (Trans40-100, 8 experiments) — mean |sim - real| / real:**

| estimator | FWHM median | FWHM IQR | sigma_fit median |
|---|---|---|---|
| ours: Lorentzian MLE | **7.3%** | 45.2% | 61.5% |
| ours: pseudo-Voigt (eta=0.2) | 83.9% | **13.8%** | **28.1%** |
| Gregor: binned Voigt LSQ | 11.2% | 20.4% | **249.6%** |

**(a) FWHM centre — best matched by our Lorentzian MLE (7.3%).** 17.8 vs 16.2 (1nW T60), 28.7 vs 28.3
(3nW T100). Gregor is ~11% low; pseudo-Voigt is ~84% high (its fixed eta=0.2 inflates the line).

**(b) FWHM spread — the estimator DOES explain it** (this is the part the original intuition got right):
Gregor's IQR is within ~20% of the real IQR and pseudo-Voigt within ~14%, whereas our unbinned MLE is ~45%
too narrow. The narrow spread was an *estimator-efficiency* artefact, not missing physics.

**(c) sigma_fit — collapse, and specifically for Gregor.** Our MLE is ~1.6x too small (the analytic CRLB
floor), but Gregor's is **~3.5x too big** (2.7-4.9 MHz vs real 0.8-1.6 MHz). That channel dominates the visual
mismatch in FIG 1: Gregor's contour sits far *above* the real dots.

**(d) Conclusion.** "Use Gregor's estimator and the simulated cloud should look like the real cloud" is
**false here**. Same estimator does not give the same cloud unless the photons, the noise, *and* the error
definition all match too.

### Lead worth checking with Gregor: our reproduction of his estimator overshoots the recorded errors ~3.5x
If `lmfit_extract` really is his procedure, its per-fit FWHM error on simulated photons should sit near the
real recorded column-2 values. It does not: e.g. 3nW T80 median 4.4 MHz vs the real 1.24 MHz. Candidates:
(i) our lmfit setup is not faithful (bin width, background model, weights, or the stderr formula);
(ii) the real per-scan spectra carry more signal than `mu_true` implies; (iii) the recorded "fit error" is
computed differently. **Confirm with Gregor before trusting the sigma_fit channel.**

### The real cloud is not reproducible by any simulator here
The real FWHM / sigma_fit distributions are **heavy-tailed**: sigma_fit has median ~1 MHz but std 4-10 MHz
(a few near-degenerate fits), and at low T the real FWHM median collapses (1nW T05: 3.7 MHz with IQR 22) —
failed fits dominate. That is a property of the experimental fitting procedure, not of our model, so the low-T
rows are excluded from the headline numbers above.

### Caveat on the "winner"
pseudo-Voigt lands closest on spread and sigma_fit, but with **eta fixed at 0.2 by hand** — so it wins partly
through a wrong-model bias that happens to widen the cloud in the right direction. Do not read it as evidence
for Voigt-like models until eta is properly fitted (or a true Voigt convolution is used).


## Notes / caveats
- Every run is generated at that experiment's **own** true `(mu, gamma, sigma_prop, lam)`. This is the
  confound that 22e removed by pinning the noise (only 1nW T60 was comparable there).
- The three estimators see the **same photons** per run (paired), so this is exactly "same data, different
  estimator".
- **`sigma_fit` is not one quantity across estimators**: ours = analytic CRLB (theory floor); Gregor's =
  lmfit covariance stderr on binned counts; the real column-2 = Gregor's recorded pipeline fit error.
- **The real cloud is heavy-tailed.** Its `sigma_fit` has median ~1 MHz but std 4-10 MHz (a few
  near-degenerate fits), and at low T the real FWHM median collapses (e.g. 1nW T05: 3.7 MHz with IQR 22) —
  failed fits dominate. So medians/IQRs are the robust comparison; the low-T subset matches no simulator.
- `ours: pseudo-Voigt` uses a **fixed eta = 0.2** mixing (not fitted) — see the discussion in the verdict.
- Raw clouds persisted to `data/processed/22g_clouds.npz`; the notebook loads them by default
  (`NB_RECOMPUTE=1` to regenerate).